In [1]:
import numpy as np

## The same delivery data, but distance is now in meters

In [2]:
X = np.array([
    [1000, 2, 0],
    [2000, 1, 1],
    [3000, 3, 0],
    [2000, 4, 1],
    [4000, 2, 0],
])

actual_times = np.array([16, 29, 28, 35, 31])

## Gradient descent from the previous blog

In [3]:
def gradient_descent(X, y, w, b, learning_rate, iterations):
    m = len(y)

    for i in range(iterations):
        predictions = X.dot(w) + b
        errors = predictions - y

        gradient_w = X.T.dot(errors) / m
        gradient_b = np.sum(errors) / m

        w = w - learning_rate * gradient_w
        b = b - learning_rate * gradient_b
        print("step:-",i,"\t", w, b)

    return w, b

## Try the old learning rate: 0.05

In [4]:
w, b = gradient_descent(
    X,
    actual_times,
    np.zeros(3),
    0.0,
    learning_rate=0.05,
    iterations=100
)

print("\n\nweights:", w)
print("bias:", b)

step:- 0 	 [3.52e+03 3.47e+00 6.40e-01] 1.3900000000000001
step:- 1 	 [-1.19679416e+09 -1.02079444e+06 -1.40798934e+05] -422397.71869999997
step:- 2 	 [4.06909170e+14 3.47069690e+11 4.78716879e+10] 143615023078.86856
step:- 3 	 [-1.38348831e+20 -1.18003450e+17 -1.62763401e+16] -4.88290065136165e+16
step:- 4 	 [4.70385048e+25 4.01210896e+22 5.53394415e+21] 1.6601827761461119e+22
step:- 5 	 [-1.59930584e+31 -1.36411422e+28 -1.88153711e+27] -5.644609724843803e+27
step:- 6 	 [5.43762858e+36 4.63797871e+33 6.39721288e+32] 1.919163323677146e+33
step:- 7 	 [-1.84878988e+42 -1.57690949e+39 -2.17504787e+38] -6.5251417591132584e+38
step:- 8 	 [6.28587255e+47 5.36148114e+44 7.39514740e+43] 2.2185435940357897e+44
step:- 9 	 [-2.13719223e+53 -1.82289980e+50 -2.51434490e+49] -7.543032565940929e+49
step:- 10 	 [7.26643851e+58 6.19784647e+55 8.54875491e+54] 2.5646257501455033e+55
step:- 11 	 [-2.47058397e+64 -2.10726343e+61 -2.90657064e+60] -8.719709454799265e+60
step:- 12 	 [8.39996805e+69 7.16468078

C:\Users\91965\AppData\Local\Temp\ipykernel_13508\2966299986.py:8: RuntimeWarning: overflow encountered in dot
  gradient_w = X.T.dot(errors) / m
C:\Users\91965\AppData\Local\Temp\ipykernel_13508\2966299986.py:8: RuntimeWarning: invalid value encountered in dot
  gradient_w = X.T.dot(errors) / m
C:\Users\91965\AppData\Local\Temp\ipykernel_13508\2966299986.py:11: RuntimeWarning: invalid value encountered in subtract
  w = w - learning_rate * gradient_w


## Find out how small the learning rate has to become

In [5]:
def test_learning_rate(X, y, learning_rate, iterations=20000):
    w = np.zeros(X.shape[1], dtype=float)
    b = 0.0
    m = len(y)

    for step in range(1, iterations + 1):
        predictions = X.dot(w) + b
        errors = predictions - y

        gradient_w = X.T.dot(errors) / m
        gradient_b = np.sum(errors) / m

        w = w - learning_rate * gradient_w
        b = b - learning_rate * gradient_b

        if not (np.all(np.isfinite(w)) and np.isfinite(b)):
            return {
                "status": "breaks",
                "step": step,
                "weights": w,
                "bias": b,
            }

    predictions = X.dot(w) + b
    cost = np.mean((predictions - y) ** 2) / 2

    return {
        "status": "survives",
        "step": iterations,
        "weights": w,
        "bias": b,
        "cost": cost,
    }

### Test the learning rates used in the blog

In [6]:
for lr in [0.0000005, 0.00000025, 0.00000032]:
    result = test_learning_rate(X.astype(float), actual_times, lr)
    print(f"learning rate = {lr}")
    print("\n\n",result)
    print()

learning rate = 5e-07


 {'status': 'breaks', 'step': 798, 'weights': array([            -inf, -2.26453306e+298, -3.12349430e+297]), 'bias': np.float64(-9.370480256259447e+297)}



C:\Users\91965\AppData\Local\Temp\ipykernel_13508\2632510968.py:10: RuntimeWarning: overflow encountered in dot
  gradient_w = X.T.dot(errors) / m


learning rate = 2.5e-07


 {'status': 'survives', 'step': 20000, 'weights': array([0.01030541, 0.04652665, 0.02253085]), 'bias': np.float64(0.014714945271921418), 'cost': np.float64(41.69878416859545)}

learning rate = 3.2e-07


 {'status': 'breaks', 'step': 4301, 'weights': array([            inf, 5.90019371e+297, 8.13819932e+296]), 'bias': np.float64(2.4414591065472522e+297)}



## Min-Max Scaling

In [7]:
def min_max_scale(values):
    v_min = min(values)
    v_max = max(values)
    return [(v - v_min) / (v_max - v_min) for v in values]

## Mean Normalization

In [8]:
def mean_normalize(values):
    v_min = min(values)
    v_max = max(values)
    v_mean = sum(values) / len(values)
    return [(v - v_mean) / (v_max - v_min) for v in values]

## Z-score Standardization

In [9]:
def z_score_scale(values):
    v_mean = sum(values) / len(values)
    variance = sum((v - v_mean) ** 2 for v in values) / len(values)
    v_std = variance ** 0.5
    return [(v - v_mean) / v_std for v in values]

## Work through one value by hand

In [10]:
distance_m = [1000, 2000, 3000, 2000, 4000]

print("Min-Max:", min_max_scale(distance_m))
print("Mean Normalization:", mean_normalize(distance_m))
print("Z-score:", z_score_scale(distance_m))

Min-Max: [0.0, 0.3333333333333333, 0.6666666666666666, 0.3333333333333333, 1.0]
Mean Normalization: [-0.4666666666666667, -0.13333333333333333, 0.2, -0.13333333333333333, 0.5333333333333333]
Z-score: [-1.3728129459672882, -0.3922322702763681, 0.5883484054145521, -0.3922322702763681, 1.5689290811054724]


## Scale the complete feature table with NumPy

The NumPy version applies Z-score standardization column by column. `axis=0` means that each feature gets its own mean and standard deviation.

In [11]:
def z_score_scale_np(x):
    return (x - x.mean(axis=0)) / x.std(axis=0)

X_scaled = z_score_scale_np(X.astype(float))

print(X_scaled)

[[-1.37281295 -0.39223227 -0.81649658]
 [-0.39223227 -1.37281295  1.22474487]
 [ 0.58834841  0.58834841 -0.81649658]
 [-0.39223227  1.56892908  1.22474487]
 [ 1.56892908 -0.39223227 -0.81649658]]


## Retrain with scaled features

In [12]:
def gradient_descent_with_history(X, y, w, b, learning_rate, iterations, checkpoints):
    m = len(y)
    history = []

    for step in range(1, iterations + 1):
        predictions = X.dot(w) + b
        errors = predictions - y

        gradient_w = X.T.dot(errors) / m
        gradient_b = np.sum(errors) / m

        w = w - learning_rate * gradient_w
        b = b - learning_rate * gradient_b

        if step in checkpoints:
            predictions = X.dot(w) + b
            cost = np.mean((predictions - y) ** 2) / 2
            history.append((step, *w, b, cost))

    return w, b, history

checkpoints = [20, 60, 100, 200, 500]

w, b, history = gradient_descent_with_history(
    X_scaled,
    actual_times,
    np.zeros(3),
    0.0,
    learning_rate=0.1,
    iterations=500,
    checkpoints=checkpoints
)

for row in history:
    print(
        f"step={row[0]:>3} | "
        f"w1={row[1]:.3f} | "
        f"w2={row[2]:.3f} | "
        f"w3={row[3]:.3f} | "
        f"b={row[4]:.3f} | "
        f"cost={row[5]:.6g}"
    )

step= 20 | w1=3.897 | w2=2.014 | w3=3.718 | b=24.420 | cost=6.68107
step= 60 | w1=5.025 | w2=2.059 | w3=4.824 | b=27.750 | cost=0.00501055
step=100 | w1=5.094 | w2=2.041 | w3=4.894 | b=27.799 | cost=1.62978e-05
step=200 | w1=5.099 | w2=2.040 | w3=4.899 | b=27.800 | cost=1.91787e-11
step=500 | w1=5.099 | w2=2.040 | w3=4.899 | b=27.800 | cost=1.08547e-28


## Check the final predictions

In [13]:
predictions = X_scaled.dot(w) + b

print("Predictions:", predictions)
print("Actual:     ", actual_times)
print("Max error:  ", np.max(np.abs(predictions - actual_times)))

Predictions: [16. 29. 28. 35. 31.]
Actual:      [16 29 28 35 31]
Max error:   2.1316282072803006e-14


## Compare the learned scaled weights with the original model

The scaled weights are not directly interpretable as minutes per kilometer/items/peak-hour anymore because the inputs are standardized. To compare them with the original coefficients, convert the standardized weights back to the original feature scale.

In [14]:
feature_means = X.astype(float).mean(axis=0)
feature_stds = X.astype(float).std(axis=0)

original_weights = w / feature_stds
original_bias = b - np.sum(original_weights * feature_means)

print("Original-scale weights:", original_weights,"\t\tprevious weights:", w)
print("Original-scale bias:", original_bias,"\t\tprevious bias:", b)

Original-scale weights: [5.e-03 2.e+00 1.e+01] 		previous weights: [5.09901951 2.03960781 4.89897949]
Original-scale bias: 7.0000000000000036 		previous bias: 27.799999999999986


## Scale a new order using the training statistics

A new prediction must use the same statistics learned from the training data. We must not calculate a fresh mean or standard deviation for the new order.

In [15]:
new_order = np.array([6000, 3, 0], dtype=float)

new_order_scaled = (new_order - feature_means) / feature_stds

new_prediction = new_order_scaled.dot(w) + b

print("Scaled new order:", new_order_scaled)
print("Predicted delivery time:", new_prediction)

Scaled new order: [ 3.53009043  0.58834841 -0.81649658]
Predicted delivery time: 42.99999999999997
